In [20]:
import gensim
import pandas as pd
from tqdm import tqdm
tqdm.pandas(desc="progress-bar")
import multiprocessing
from sklearn import utils
from gensim.models import Doc2Vec
from clean_text import clean_text
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import TaggedDocument
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [74]:
train_df = pd.read_csv("train.csv")
copy_train_df = train_df.copy()
copy_train_df.drop(["id", "discourse_id", "discourse_start", "discourse_end", "discourse_type_num", "predictionstring"], axis=1, inplace=True)

In [75]:
discourse_text = []
discourse_type = []
for i,paragraph in enumerate(copy_train_df.discourse_text):
    types = []
    phrases = sent_tokenize(paragraph)
    types.append(copy_train_df.discourse_type[i])
    discourse_text.extend(phrases)
    discourse_type.extend(types*len(phrases))

In [79]:
copy_train_df = pd.DataFrame({"discourse_text":discourse_text, "discourse_type":discourse_type})

In [80]:
copy_train_df["id"]=copy_train_df.discourse_type.index.to_numpy()

In [81]:
copy_train_df.discourse_text = copy_train_df.discourse_text.apply(clean_text)

In [106]:
copy_train_df.head()

,discourse_text,discourse_type,id
0,modern human today always phone,Lead,0
1,always phone nbr hour day stop text back forwa...,Lead,1
2,even driving,Lead,2
3,really bad consequence stuff happens come phone,Position,3
4,certain area united state ban phone class room,Evidence,4


In [107]:
train, test = train_test_split(copy_train_df, test_size=0.2, random_state=42)

In [108]:
train_tagged = train.apply(lambda x: TaggedDocument(words=[word for word in word_tokenize(x["discourse_text"])], tags=[x.discourse_type]), axis=1)
test_tagged = test.apply(lambda x: TaggedDocument(words=[word for word in word_tokenize(x["discourse_text"])], tags=[x.discourse_type]), axis=1)

In [109]:
train_tagged.values[30]

TaggedDocument(words=['dont', 'sit', 'taking', 'care', 'horse', 'farm', 'animal', 'also', 'get', 'play', 'game'], tags=['Evidence'])

In [110]:
cores = multiprocessing.cpu_count()

In [111]:
model_dbow = Doc2Vec(dm=0, vector_size=300, negative=5, hs=0, min_count=2, sample = 0, workers=cores)
model_dbow.build_vocab([x for x in tqdm(train_tagged.values)])

100%|██████████| 264979/264979 [00:01<00:00, 159083.81it/s]


In [112]:
%%time
for epoch in range(30):
    model_dbow.train(utils.shuffle([x for x in tqdm(train_tagged.values)]), total_examples=len(train_tagged.values), epochs=1)
    model_dbow.alpha -= 0.002
    model_dbow.min_alpha = model_dbow.alpha

100%|██████████| 264979/264979 [00:00<00:00, 3030351.10it/s]


Wall time: 6min 1s


In [89]:
def vec_for_learning(model, tagged_docs):
    sents = tagged_docs.values
    targets, regressors = zip(*[(doc.tags[0], model.infer_vector(doc.words)) for doc in sents])
    return targets, regressors

In [97]:
from scipy import spatial

In [120]:
a = model_dbow.infer_vector(word_tokenize(copy_train_df.discourse_text[1]))
b = model_dbow.infer_vector(word_tokenize(copy_train_df.discourse_text[125]))
spatial.distance.cosine(a,b)

0.005807757377624512

In [90]:
y_train, X_train = vec_for_learning(model_dbow, train_tagged)
y_test, X_test = vec_for_learning(model_dbow, test_tagged)

In [69]:
y_train[50]

'Evidence'

In [91]:
logreg = LogisticRegression(n_jobs=1, C=1e5)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

C:\Users\user\Anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


In [92]:
print('Testing accuracy %s' % accuracy_score(y_test, y_pred))
print('Testing F1 score: {}'.format(f1_score(y_test, y_pred, average='weighted')))

Testing accuracy 0.5256396709185599
Testing F1 score: 0.367796934612748


In [93]:
print(classification_report(y_test, y_pred))

C:\Users\user\Anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\user\Anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


                      precision    recall  f1-score   support

               Claim       0.45      0.00      0.01     11041
Concluding Statement       0.73      0.02      0.03      8372
        Counterclaim       0.19      0.01      0.01      1476
            Evidence       0.53      1.00      0.69     34711
                Lead       0.00      0.00      0.00      5691
            Position       0.21      0.01      0.02      3621
            Rebuttal       0.00      0.00      0.00      1333

            accuracy                           0.53     66245
           macro avg       0.30      0.15      0.11     66245
        weighted avg       0.46      0.53      0.37     66245



C:\Users\user\Anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## MultinomialNB

In [32]:
import xgboost as xgb

In [34]:
xgb_model = xgb.XGBClassifier(max_depth=50, n_estimators=80, learning_rate=0.1, colsample_bytree=.7, gamma=0, reg_alpha=4, objective='multi:softprob', eta=0.3, silent=1, subsample=0.8)
xgb_model.fit(X_train, y_train)

[15:39:26] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:576: 
Parameters: { "silent" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


[15:44:52] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'multi:softprob' was changed from 'merror' to 'mlogloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [ ]:
xgb_prediction = xgb_model.predict(X_test)
print('character level tf-idf training score:', f1_score(y_train, xgb_model.predict(X_train), average='macro'))
print('character level tf-idf validation score:', f1_score(y_test, xgb_model.predict(X_test), average='macro'))
print(classification_report(y_test, xgb_prediction))